# CordiaAI — Step 3: GAN Augmentation (Colab / GPU run)

Self-contained version of the local `run_augmentation.py` pipeline, with epoch counts raised for a real (not quick-sanity-check) comparison of SMOTE vs CTGAN vs a diffusion-based generator.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

This notebook re-downloads and re-preprocesses the UCI data itself (same logic as the local `src/preprocessing/` modules) so it doesn't depend on any local files.

In [ ]:
!pip install -q ctgan imbalanced-learn
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (enable GPU in Runtime settings)')

## Step 1 — Data collection (same as local `load_data.py`)

In [ ]:
import pandas as pd

_BASE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease"
_SOURCES = ["cleveland", "hungarian", "switzerland", "va"]
_COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
]

frames = []
for source in _SOURCES:
    url = f"{_BASE_URL}/processed.{source}.data"
    df = pd.read_csv(url, header=None, names=_COLUMNS, na_values="?")
    df["source"] = source
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
data["num"] = data["num"].astype(int)
print(f"Loaded {len(data)} rows, {data.shape[1]} columns")
print(data["source"].value_counts())

## Step 2 — Preprocessing (same as local `clean.py` / `split.py`, leak-free)

In [ ]:
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

TARGET_COL = "num"
N_SF2_FEATURES = 10

def select_sf2(X, y, k=N_SF2_FEATURES):
    X_nonneg = MinMaxScaler().fit_transform(X)
    anova_scores = f_classif(X, y)[0]
    chi2_scores = chi2(X_nonneg, y)[0]
    mi_scores = mutual_info_classif(X, y, random_state=42)
    ranks = pd.DataFrame({
        "anova_rank": pd.Series(-anova_scores).rank(),
        "chi2_rank": pd.Series(-chi2_scores).rank(),
        "mi_rank": pd.Series(-mi_scores).rank(),
    }, index=X.columns)
    ranks["consensus"] = ranks.sum(axis=1)
    return ranks.sort_values("consensus").head(k).index.tolist()

class Preprocessor:
    def __init__(self, k=N_SF2_FEATURES):
        self.k = k
        self.medians_ = None
        self.sf2_features_ = None
        self.scaler_ = None

    def fit(self, X, y):
        numeric_cols = X.select_dtypes(include="number").columns
        self.medians_ = X[numeric_cols].median()
        X_imputed = X[numeric_cols].fillna(self.medians_)
        self.sf2_features_ = select_sf2(X_imputed, y, self.k)
        self.scaler_ = StandardScaler().fit(X_imputed[self.sf2_features_])
        return self

    def transform(self, X):
        X_imputed = X[self.medians_.index].fillna(self.medians_)
        X_selected = X_imputed[self.sf2_features_]
        return pd.DataFrame(self.scaler_.transform(X_selected), columns=self.sf2_features_, index=X.index)

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

y = data[TARGET_COL]
X = data.drop(columns=[TARGET_COL]).select_dtypes(include="number")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

prep = Preprocessor()
X_train_p = prep.fit_transform(X_train, y_train)
X_test_p = prep.transform(X_test)
print(f"SF-2 features: {prep.sf2_features_}")
print(f"Train: {X_train_p.shape}  Test: {X_test_p.shape}")
print(y_train.value_counts().sort_index())

## Step 3 — Evaluation metrics (same as local `evaluate.py`)

In [ ]:
import numpy as np
from scipy import linalg
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import NearestNeighbors

def frechet_distance(real, synthetic):
    mu1, mu2 = real.mean().values, synthetic.mean().values
    sigma1, sigma2 = np.cov(real.values, rowvar=False), np.cov(synthetic.values, rowvar=False)
    diff = mu1 - mu2
    covmean = linalg.sqrtm(sigma1 @ sigma2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean))

def ml_efficacy(X_train_real, y_train_real, X_synth, y_synth, X_test, y_test, random_state=42):
    clf_real = RandomForestClassifier(random_state=random_state).fit(X_train_real, y_train_real)
    real_only_acc = accuracy_score(y_test, clf_real.predict(X_test))
    real_only_f1 = f1_score(y_test, clf_real.predict(X_test), average="macro")
    X_combined = pd.concat([X_train_real, X_synth], ignore_index=True)
    y_combined = pd.concat([y_train_real, y_synth], ignore_index=True)
    clf_aug = RandomForestClassifier(random_state=random_state).fit(X_combined, y_combined)
    aug_acc = accuracy_score(y_test, clf_aug.predict(X_test))
    aug_f1 = f1_score(y_test, clf_aug.predict(X_test), average="macro")
    return {"real_only_accuracy": real_only_acc, "real_only_macro_f1": real_only_f1,
            "augmented_accuracy": aug_acc, "augmented_macro_f1": aug_f1}

def nearest_neighbour_privacy_distance(real, synthetic):
    nn = NearestNeighbors(n_neighbors=1).fit(real.values)
    distances, _ = nn.kneighbors(synthetic.values)
    return {"min_distance": float(distances.min()), "mean_distance": float(distances.mean()),
            "5th_percentile_distance": float(np.percentile(distances, 5))}

## The three generators (same logic as local `smote_baseline.py` / `ctgan_generator.py` / `diffusion_generator.py`)

In [ ]:
from imblearn.over_sampling import SMOTE
from ctgan import CTGAN
from torch import nn

def smote_augment(X_train, y_train, target_counts, random_state=42):
    smallest = y_train.value_counts().min()
    k_neighbors = max(1, min(5, smallest - 1))
    sampling_strategy = {cls: n for cls, n in target_counts.items() if n > (y_train == cls).sum()}
    if not sampling_strategy:
        return X_train.iloc[0:0], y_train.iloc[0:0]
    smote = SMOTE(sampling_strategy=sampling_strategy, k_neighbors=k_neighbors, random_state=random_state)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    n = len(X_train)
    return X_res.iloc[n:].reset_index(drop=True), y_res.iloc[n:].reset_index(drop=True)

def ctgan_augment(X_train, y_train, target_counts, epochs=800, random_state=42):
    X_parts, y_parts = [], []
    for cls, target in target_counts.items():
        rows = X_train[y_train == cls]
        deficit = target - len(rows)
        if deficit <= 0:
            continue
        model = CTGAN(epochs=epochs, batch_size=10, pac=1, cuda=torch.cuda.is_available(), verbose=False)
        model.fit(rows)
        X_parts.append(model.sample(deficit))
        y_parts.append(pd.Series([cls] * deficit))
    if not X_parts:
        return X_train.iloc[0:0], y_train.iloc[0:0]
    return pd.concat(X_parts, ignore_index=True), pd.concat(y_parts, ignore_index=True)

class _Denoiser(nn.Module):
    def __init__(self, n_features, hidden=128):
        super().__init__()
        self.time_embed = nn.Embedding(1000, hidden)
        self.net = nn.Sequential(nn.Linear(n_features + hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_features))
    def forward(self, x, t):
        return self.net(torch.cat([x, self.time_embed(t)], dim=1))

class TabularDiffusion:
    def __init__(self, n_features, timesteps=200, random_state=42, device=None):
        torch.manual_seed(random_state)
        self.timesteps = timesteps
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model = _Denoiser(n_features).to(self.device)
        self.betas = torch.linspace(1e-4, 0.02, timesteps, device=self.device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def fit(self, X, epochs=800, lr=1e-3):
        x0 = torch.tensor(X.values, dtype=torch.float32, device=self.device)
        opt = torch.optim.Adam(self.model.parameters(), lr=lr)
        for _ in range(epochs):
            t = torch.randint(0, self.timesteps, (len(x0),), device=self.device)
            noise = torch.randn_like(x0)
            ab = self.alpha_bars[t].unsqueeze(1)
            x_t = ab.sqrt() * x0 + (1 - ab).sqrt() * noise
            loss = nn.functional.mse_loss(self.model(x_t, t), noise)
            opt.zero_grad(); loss.backward(); opt.step()
        return self

    @torch.no_grad()
    def sample(self, n, n_features):
        x = torch.randn(n, n_features, device=self.device)
        for t in reversed(range(self.timesteps)):
            t_batch = torch.full((n,), t, dtype=torch.long, device=self.device)
            alpha_t, alpha_bar_t, beta_t = self.alphas[t], self.alpha_bars[t], self.betas[t]
            pred_noise = self.model(x, t_batch)
            mean = (1 / alpha_t.sqrt()) * (x - (1 - alpha_t) / (1 - alpha_bar_t).sqrt() * pred_noise)
            x = mean + beta_t.sqrt() * torch.randn_like(x) if t > 0 else mean
        return x.cpu()

def diffusion_augment(X_train, y_train, target_counts, epochs=800, random_state=42):
    X_parts, y_parts = [], []
    n_features = X_train.shape[1]
    for cls, target in target_counts.items():
        rows = X_train[y_train == cls]
        deficit = target - len(rows)
        if deficit <= 0:
            continue
        diffusion = TabularDiffusion(n_features, random_state=random_state).fit(rows, epochs=epochs)
        samples = diffusion.sample(deficit, n_features).numpy()
        X_parts.append(pd.DataFrame(samples, columns=X_train.columns))
        y_parts.append(pd.Series([cls] * deficit))
    if not X_parts:
        return X_train.iloc[0:0], y_train.iloc[0:0]
    return pd.concat(X_parts, ignore_index=True), pd.concat(y_parts, ignore_index=True)

## Run the comparison — epochs raised from the local quick-run's 100 to 800
On a T4 GPU this should take a few minutes total, versus a much longer CPU run at the same epoch count.

In [ ]:
EPOCHS = 800

counts = y_train.value_counts().sort_index()
target = int(counts.sort_values(ascending=False).iloc[1])
target_counts = {cls: target for cls in [2, 3, 4]}
print(f"Train class counts:\n{counts}\n\nAugmentation target per minority class: {target}\n")

generators = {
    "SMOTE": lambda: smote_augment(X_train_p, y_train, target_counts),
    "CTGAN": lambda: ctgan_augment(X_train_p, y_train, target_counts, epochs=EPOCHS),
    "Diffusion": lambda: diffusion_augment(X_train_p, y_train, target_counts, epochs=EPOCHS),
}

results = {}
synthetic_sets = {}
for name, generate in generators.items():
    print(f"--- {name} ---")
    X_synth, y_synth = generate()
    synthetic_sets[name] = (X_synth, y_synth)
    if len(X_synth) == 0:
        print("No synthetic samples generated, skipping.\n")
        continue
    fd = frechet_distance(X_train_p, X_synth)
    privacy = nearest_neighbour_privacy_distance(X_train_p, X_synth)
    efficacy = ml_efficacy(X_train_p, y_train, X_synth, y_synth, X_test_p, y_test)
    results[name] = {"frechet_distance": fd, **privacy, **efficacy}
    for k, v in results[name].items():
        print(f"  {k}: {v:.4f}")
    print()

print("=" * 60)
print("Summary (lower FD better; higher privacy dist & aug acc/F1 better)")
print("=" * 60)
for name, m in results.items():
    print(f"{name:10s} FD={m['frechet_distance']:.3f}  privacy(min)={m['min_distance']:.3f}  "
          f"aug_acc={m['augmented_accuracy']:.3f}  aug_f1={m['augmented_macro_f1']:.3f}")

## Save results
Downloads the best generator's synthetic data + the comparison table so you can bring them back to the local repo.

In [ ]:
pd.DataFrame(results).T.to_csv("augmentation_comparison.csv")

# Pick the winning generator by editing this line once you've looked at the table above:
BEST = "CTGAN"
X_best, y_best = synthetic_sets[BEST]
X_best.assign(num=y_best).to_csv("best_synthetic_samples.csv", index=False)

from google.colab import files
files.download("augmentation_comparison.csv")
files.download("best_synthetic_samples.csv")